# Neural Speech Decoding — Uncertainty Estimation Demo

This notebook demonstrates the full uncertainty estimation pipeline for the neural speech decoding system.
All core logic lives in `src/`; this notebook only handles configuration, calls, and visualisation.

**Pipeline overview:**
```
Neural signal (ECoG)
    │
    ▼
DBConformerCTC  (acoustic model)  → phoneme sequence + AM uncertainty
    │
    ▼
BART-large or Qwen2.5-7B          → text + LM uncertainty
    │
    ▼
Calibration analysis, ECE, coverage–WER trade-off
```

## 0. Setup

In [ ]:
# If running on Google Colab, mount Drive and install dependencies first:
# from google.colab import drive
# drive.mount('/content/drive')
# !pip install -q evaluate jiwer rouge_score bitsandbytes>=0.46.1 peft trl einops timm

import argparse
import pickle
import torch

# Add the repo root to sys.path if needed
import sys, os
sys.path.insert(0, os.path.abspath('..'))

# Import everything from src
from src import (
    build_dataloader, load_acoustic_model,
    load_bart, load_qwen,
    evaluate_am_uncertainty,
    evaluate_pipeline_uncertainty_bart,
    evaluate_pipeline_uncertainty_qwen,
    evaluate_ece, plot_coverage_wer_tradeoff,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Configuration
Adjust paths here — everything else is handled by `src/`.

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
VAL_INDEX_PATH      = '/content/val_index_merged.pkl'
ACOUSTIC_CKPT_PATH  = '/content/drive/MyDrive/DBConformer_runs_512/NeuralPhoneme/DBConformerCTC_t15.2023.08.18_best.ckpt'
BART_MODEL_PATH     = '/content/drive/MyDrive/SpeechModels/BART_Phoneme_to_Text_Final_BART_large_Synthetic'
QWEN_BASE_ID        = 'Qwen/Qwen2.5-7B-Instruct'
QWEN_LORA_PATH      = '/content/drive/MyDrive/SpeechModels/Qwen2.5_Phoneme_to_Text_LoRA_syntactic'

# ── Model hyperparameters (must match training config) ───────────────────────
args = argparse.Namespace(
    data_name            = 'NeuralPhoneme',
    chn                  = 512,
    time_sample_num      = 1500,
    emb_size             = 512,
    spa_dim              = 16,
    transformer_depth_tem = 6,
    transformer_depth_chn = 6,
    temporal_kernel      = 15,
    pool_kernel          = 15,
    pool_stride          = 8,
    gate_flag            = False,
    posemb_flag          = True,
    chn_atten_flag       = True,
    branch               = 'all',
    F1=4, D=2, F2=8,
    ffn_hidden           = 1024,
    dropoutRate          = 0.3,
    class_num            = 41,
    max_time_steps       = 1500,
)

BATCH_SIZE = 16   # increase if VRAM > 22 GB

## 2. Load Data

In [ ]:
with open(VAL_INDEX_PATH, 'rb') as f:
    val_index = pickle.load(f)

val_loader = build_dataloader(
    val_index,
    batch_size=BATCH_SIZE,
    max_time_steps=args.max_time_steps,
    with_text=True,
)
print(f'Validation samples: {len(val_loader.dataset)}')

## 3. Load Acoustic Model

In [ ]:
acoustic_model = load_acoustic_model(args, ACOUSTIC_CKPT_PATH, device)

## 4. Acoustic Model Uncertainty (standalone)
Evaluates how well `U_AM` predicts phoneme error rate, without any language model.

In [ ]:
# Note: evaluate_am_uncertainty expects a loader WITHOUT text labels.
# Build a separate loader with with_text=False:
val_loader_no_text = build_dataloader(
    val_index, batch_size=BATCH_SIZE,
    max_time_steps=args.max_time_steps, with_text=False
)

am_pers, am_uncs = evaluate_am_uncertainty(
    acoustic_model, val_loader_no_text, device, args
)

## 5. BART Pipeline Uncertainty

In [ ]:
lm_model, lm_tokenizer = load_bart(BART_MODEL_PATH, device)

bart_wers, bart_pers, bart_am_uncs, bart_lm_uncs = evaluate_pipeline_uncertainty_bart(
    acoustic_model, lm_model, lm_tokenizer, val_loader, device, args
)

print('\n--- BART ECE ---')
evaluate_ece(bart_wers, bart_lm_uncs)

plot_coverage_wer_tradeoff(bart_wers, bart_am_uncs, bart_lm_uncs,
                            save_path='bart_coverage_vs_wer.png')

## 6. Qwen Pipeline Uncertainty

> ⚠️  Requires ~14–19 GB VRAM.  
> Free BART memory first if you are on a shared GPU.

In [ ]:
# Free BART from GPU memory before loading Qwen
import gc
del lm_model
torch.cuda.empty_cache()
gc.collect()
print('BART freed.')

In [ ]:
# use_quantization=True  → 4-bit NF4, ~5 GB VRAM for Qwen (recommended on ≤16 GB GPUs)
# use_quantization=False → bfloat16, ~14 GB VRAM, faster inference
qwen_model, qwen_tokenizer = load_qwen(
    QWEN_BASE_ID, QWEN_LORA_PATH, use_quantization=True
)

qwen_wers, qwen_pers, qwen_am_uncs, qwen_lm_uncs = evaluate_pipeline_uncertainty_qwen(
    acoustic_model, qwen_model, qwen_tokenizer, val_loader, device, args
)

print('\n--- Qwen ECE ---')
evaluate_ece(qwen_wers, qwen_lm_uncs)

plot_coverage_wer_tradeoff(qwen_wers, qwen_am_uncs, qwen_lm_uncs,
                            save_path='qwen_coverage_vs_wer.png')

## 7. Comparative Summary

In [ ]:
import numpy as np

print('=' * 55)
print(f'{"Metric":<35} {"BART":>8} {"Qwen":>8}')
print('=' * 55)

rows = [
    ('Mean WER',              np.mean(bart_wers),    np.mean(qwen_wers)),
    ('Mean PER (AM)',         np.mean(bart_pers),    np.mean(qwen_pers)),
    ('AM unc vs PER (corr)',  np.corrcoef(bart_am_uncs, bart_pers)[0,1],
                              np.corrcoef(qwen_am_uncs, qwen_pers)[0,1]),
    ('AM unc vs WER (corr)',  np.corrcoef(bart_am_uncs, bart_wers)[0,1],
                              np.corrcoef(qwen_am_uncs, qwen_wers)[0,1]),
    ('LM unc vs WER (corr)',  np.corrcoef(bart_lm_uncs, bart_wers)[0,1],
                              np.corrcoef(qwen_lm_uncs, qwen_wers)[0,1]),
    ('LM unc vs PER (corr)',  np.corrcoef(bart_lm_uncs, bart_pers)[0,1],
                              np.corrcoef(qwen_lm_uncs, qwen_pers)[0,1]),
]

for name, b_val, q_val in rows:
    print(f'{name:<35} {b_val:>8.3f} {q_val:>8.3f}')
print('=' * 55)